#  Data Engineering & Machine Learning avec Snowflake
## Prédiction du Prix des Maisons

**Dataset :** House Price Prediction  
**Source :** `s3://logbrain-datalake/datasets/house_price/`  
**Objectif :** Construire un pipeline ML complet dans Snowflake pour prédire le prix des maisons.

---

## Étape 1 — Configuration de l'environnement

In [ ]:

import snowflake.snowpark as snowpark
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


session = get_active_session()

print(f"✅ Connecté à Snowflake")
print(f"   Account  : {session.get_current_account()}")
print(f"   Role     : {session.get_current_role()}")
print(f"   Warehouse: {session.get_current_warehouse()}")
print(f"   Database : {session.get_current_database()}")
print(f"   Schema   : {session.get_current_schema()}")

## Étape 2 — Ingestion des données depuis S3

In [ ]:
-- Création du stage externe pointant vers le bucket S3
CREATE OR REPLACE STAGE HOUSE_PRICE_STAGE
    URL = 's3://logbrain-datalake/datasets/house_price/'
    FILE_FORMAT = (
        TYPE = 'JSON'
        STRIP_OUTER_ARRAY = TRUE
    );

In [ ]:
-- Vérification du contenu du stage
LIST @HOUSE_PRICE_STAGE;

In [ ]:
-- Création de la table
CREATE OR REPLACE TABLE HOUSE_PRICE (
    PRICE            FLOAT,
    AREA             FLOAT,
    BEDROOMS         INT,
    BATHROOMS        INT,
    STORIES          INT,
    MAINROAD         VARCHAR(3),
    GUESTROOM        VARCHAR(3),
    BASEMENT         VARCHAR(3),
    HOTWATERHEATING  VARCHAR(3),
    AIRCONDITIONING  VARCHAR(3),
    PARKING          INT,
    PREFAREA         VARCHAR(3),
    FURNISHINGSTATUS VARCHAR(20)
);

In [ ]:
-- Chargement des données depuis le stage 
COPY INTO HOUSE_PRICE (PRICE, AREA, BEDROOMS, BATHROOMS, STORIES, MAINROAD,
                        GUESTROOM, BASEMENT, HOTWATERHEATING, AIRCONDITIONING,
                        PARKING, PREFAREA, FURNISHINGSTATUS)
FROM (
    SELECT
        $1:price::FLOAT,
        $1:area::FLOAT,
        $1:bedrooms::INT,
        $1:bathrooms::INT,
        $1:stories::INT,
        $1:mainroad::VARCHAR,
        $1:guestroom::VARCHAR,
        $1:basement::VARCHAR,
        $1:hotwaterheating::VARCHAR,
        $1:airconditioning::VARCHAR,
        $1:parking::INT,
        $1:prefarea::VARCHAR,
        $1:furnishingstatus::VARCHAR
    FROM @HOUSE_PRICE_STAGE
)
FILE_FORMAT = (TYPE = 'JSON' STRIP_OUTER_ARRAY = TRUE)
ON_ERROR = 'CONTINUE';

In [ ]:
-- Vérification du chargement
SELECT COUNT(*) AS NB_LIGNES FROM HOUSE_PRICE;

## Étape 3 — Exploration des données


In [ ]:
df_pd = session.table("HOUSE_PRICE").to_pandas()

print(f"Dimensions : {df_pd.shape[0]} lignes x {df_pd.shape[1]} colonnes")
df_pd.head(10)

In [ ]:
# Statistiques descriptives 
df_pd.describe().round(2)

In [ ]:
# Vérification des valeurs nulles
null_counts = df_pd.isnull().sum()
print("Valeurs nulles par colonne :")
print(null_counts[null_counts >= 0].to_string())

In [ ]:
# Distribution des variables catégorielles vs prix
cat_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING',
            'AIRCONDITIONING', 'PREFAREA', 'FURNISHINGSTATUS']
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df_pd.groupby(col)['PRICE'].agg(['count', 'mean', 'median']).round(0))

In [ ]:
# Prix moyen selon combinaison d'équipements clés
combo = (df_pd.groupby(['AIRCONDITIONING', 'PREFAREA', 'BASEMENT'])
              .agg(nb_maisons=('PRICE', 'count'), prix_moyen=('PRICE', 'mean'))
              .round(0)
              .sort_values('prix_moyen', ascending=False)
              .head(10))
print(combo.to_string())

In [ ]:
# Visualisations 

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_pd['PRICE'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution du Prix', fontsize=13)
axes[0].set_xlabel('Prix')
axes[0].set_ylabel('Fréquence')

axes[1].hist(np.log1p(df_pd['PRICE']), bins=40, color='coral', edgecolor='white')
axes[1].set_title('Distribution du Log(Prix)', fontsize=13)
axes[1].set_xlabel('Log(Prix)')

plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation
numeric_cols = ['PRICE', 'AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES', 'PARKING']
corr_matrix  = df_pd[numeric_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Matrice de Corrélation', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Prix médian par variable catégorielle
cat_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 'AIRCONDITIONING', 'PREFAREA', 'FURNISHINGSTATUS']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    df_pd.groupby(col)['PRICE'].median().plot(kind='bar', ax=axes[i],
                                               color='steelblue', edgecolor='white')
    axes[i].set_title(f'Prix médian par {col}', fontsize=11)
    axes[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Étape 4 — Préparation des données (Feature Engineering)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# df_pd est déjà chargé depuis l'étape 3
df_clean = df_pd.copy()

# Encodage binaire yes/no -> 1/0
binary_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING', 'AIRCONDITIONING', 'PREFAREA']
for col in binary_cols:
    df_clean[col] = df_clean[col].map({'yes': 1, 'no': 0})

# Encodage ordinal de FURNISHINGSTATUS
df_clean['FURNISHINGSTATUS'] = df_clean['FURNISHINGSTATUS'].map({
    'furnished'      : 2,
    'semi-furnished' : 1,
    'unfurnished'    : 0
})

print("Encodage effectué")
print(df_clean.head())

In [ ]:
# Séparation features / cible + split train/test reproductible
FEATURES = ['AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES', 'MAINROAD',
            'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING', 'AIRCONDITIONING',
            'PARKING', 'PREFAREA', 'FURNISHINGSTATUS']
TARGET = 'PRICE'

X = df_clean[FEATURES]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalisation (pour les modèles linéaires)
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Split effectué — Train : {len(X_train)} lignes | Test : {len(X_test)} lignes")

## Étape 5 — Entraînement des modèles

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled
# sont déjà disponibles depuis l'étape 4
print(f"Données prêtes — Train : {len(X_train)} lignes | Test : {len(X_test)} lignes")

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2   = r2_score(y_te, y_pred)
    print(f"\n{name}")
    print(f"   MAE  : {mae:,.0f}")
    print(f"   RMSE : {rmse:,.0f}")
    print(f"   R²   : {r2:.4f}")
    return {'name': name, 'model': model, 'mae': mae, 'rmse': rmse, 'r2': r2}

results = [
    evaluate_model(LinearRegression(),
                   X_train_scaled, y_train, X_test_scaled, y_test, "Linear Regression"),
    evaluate_model(Ridge(alpha=1.0),
                   X_train_scaled, y_train, X_test_scaled, y_test, "Ridge Regression"),
    evaluate_model(RandomForestRegressor(n_estimators=100, random_state=42),
                   X_train, y_train, X_test, y_test, "Random Forest"),
    evaluate_model(XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
                   X_train, y_train, X_test, y_test, "XGBoost"),
]

print("\n Entraînement terminé")

## Étape 6 — Évaluation et comparaison

In [ ]:
# Tableau comparatif
results_df = pd.DataFrame([{
    'Modèle': r['name'],
    'MAE'   : round(r['mae'], 0),
    'RMSE'  : round(r['rmse'], 0),
    'R²'    : round(r['r2'], 4)
} for r in results]).sort_values('R²', ascending=False).reset_index(drop=True)

print(results_df.to_string(index=False))

In [ ]:
# Graphiques comparatifs
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
names  = results_df['Modèle']

for ax, metric in zip(axes, ['MAE', 'RMSE', 'R²']):
    ax.bar(names, results_df[metric], color=colors)
    ax.set_title(f'{metric}', fontsize=12)
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# Importance des features (Random Forest)
rf_model = next(r['model'] for r in results if r['name'] == 'Random Forest')
feat_imp = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values()

plt.figure(figsize=(9, 6))
feat_imp.plot(kind='barh', color='steelblue')
plt.title('Importance des Features (Random Forest)', fontsize=13)
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Étape 7 — Optimisation avec Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint

# GridSearchCV — XGBoost
# Grille réduite pour limiter le nombre de fits
print(" GridSearchCV sur XGBoost...")
xgb_gs = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    param_grid={
        'n_estimators' : [100, 200],
        'max_depth'    : [3, 5],
        'learning_rate': [0.05, 0.1],

    },
    cv=3, scoring='r2', n_jobs=1, verbose=1
)
xgb_gs.fit(X_train, y_train)
print(f"\n Meilleurs paramètres : {xgb_gs.best_params_}")
print(f"   R² CV : {xgb_gs.best_score_:.4f}")

In [ ]:
# RandomizedSearchCV — Random Forest
print("🔍 RandomizedSearchCV sur Random Forest...")
rf_rs = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions={
        'n_estimators'     : randint(50, 200),
        'max_depth'        : [None, 5, 10, 15],
        'min_samples_split': randint(2, 8),
        'max_features'     : ['sqrt', 'log2']
    },
    n_iter=10, cv=3, scoring='r2', n_jobs=1, random_state=42, verbose=1
)
rf_rs.fit(X_train, y_train)
print(f"\n Meilleurs paramètres : {rf_rs.best_params_}")
print(f"   R² CV : {rf_rs.best_score_:.4f}")

In [ ]:
# Évaluation des modèles optimisés et sélection du meilleur
optimized = []
for name, model in [('XGBoost (GridSearch)', xgb_gs.best_estimator_),
                     ('Random Forest (RandomSearch)', rf_rs.best_estimator_)]:
    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    optimized.append({'name': name, 'model': model, 'mae': mae, 'rmse': rmse, 'r2': r2})
    print(f"\n {name}")
    print(f"   MAE={mae:,.0f} | RMSE={rmse:,.0f} | R²={r2:.4f}")

best_result = max(optimized, key=lambda x: x['r2'])
best_model  = best_result['model']
print(f"\n Meilleur modèle : {best_result['name']} (R²={best_result['r2']:.4f})")

## Étape 8 — Stockage dans le Snowflake Model Registry

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session=session)

# Données de test au format Snowpark pour l'échantillon d'entrée
X_test_sp = session.create_dataframe(pd.DataFrame(X_test, columns=FEATURES))

model_version = registry.log_model(
    model=best_model,
    model_name="HOUSE_PRICE_MODEL",
    version_name="v1",
    comment=f"Meilleur modèle : {best_result['name']}",
    metrics={
        'MAE' : round(best_result['mae'], 2),
        'RMSE': round(best_result['rmse'], 2),
        'R2'  : round(best_result['r2'], 4)
    },
    sample_input_data=X_test_sp
)

print(f"Modèle enregistré : HOUSE_PRICE_MODEL / v1")
print(f"   MAE={best_result['mae']:,.0f} | RMSE={best_result['rmse']:,.0f} | R²={best_result['r2']:.4f}")

In [ ]:
-- Vérification du modèle dans le registry
SHOW MODELS LIKE 'HOUSE_PRICE_MODEL';

In [ ]:
-- Affichage des versions disponibles
SHOW VERSIONS IN MODEL HOUSE_PRICE_MODEL;

## Étape 9 — Inférence avec le modèle depuis le Registry

In [ ]:
# Chargement du modèle depuis le registry
loaded_model = registry.get_model("HOUSE_PRICE_MODEL").version("v1")
print("Modèle chargé depuis le registry")

# Inférence sur le jeu de test
predictions_sp = loaded_model.run(X_test_sp, function_name="predict")
predictions_sp.show(10)

# Sauvegarde des prédictions dans une table Snowflake
predictions_sp.write.mode("overwrite").save_as_table("HOUSE_PRICE_PREDICTIONS")
print("Prédictions sauvegardées dans HOUSE_PRICE_PREDICTIONS")

In [ ]:
-- Vérification des prédictions stockées
SELECT * FROM HOUSE_PRICE_PREDICTIONS LIMIT 10;

In [ ]:
# Analyse des écarts prédictions vs réalité
y_pred_final = best_model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred_final)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
r2   = r2_score(y_test, y_pred_final)

print(f"MAE        : {mae:,.0f}")
print(f"RMSE       : {rmse:,.0f}")
print(f"R²         : {r2:.4f}")
print(f"Pred. min  : {y_pred_final.min():,.0f}")
print(f"Pred. max  : {y_pred_final.max():,.0f}")
print(f"Pred. moy  : {y_pred_final.mean():,.0f}")

In [ ]:
# Visualisation prédictions vs réalité
y_pred_final = best_model.predict(X_test)

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_final, alpha=0.6, color='steelblue', edgecolors='white', s=50)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', lw=2, label='Prédiction parfaite')
plt.xlabel('Prix réel', fontsize=12)
plt.ylabel('Prix prédit', fontsize=12)
plt.title(f'Prédictions vs Valeurs Réelles\n{best_result["name"]} — R²={best_result["r2"]:.4f}', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

## Étape 10 — Application Streamlit dans Snowflake

> Copier le code ci-dessous dans **Snowflake > Streamlit > + New Streamlit App**

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

st.set_page_config(page_title="🏠 House Price Predictor", layout="wide")

@st.cache_resource
def get_session():
    return get_active_session()

@st.cache_resource
def load_model(_session):
    registry = Registry(session=_session)
    return registry.get_model("HOUSE_PRICE_MODEL").version("v1")

session = get_session()
model   = load_model(session)

st.title("🏠 Prédiction du Prix des Maisons")
st.markdown("Renseignez les caractéristiques pour obtenir une estimation du prix.")
st.divider()

col1, col2, col3 = st.columns(3)

with col1:
    st.subheader("Caractéristiques physiques")
    area      = st.number_input("Surface (m²)", min_value=500, max_value=20000, value=5000, step=100)
    bedrooms  = st.slider("Chambres", 1, 6, 3)
    bathrooms = st.slider("Salles de bain", 1, 4, 1)
    stories   = st.slider("Étages", 1, 4, 1)
    parking   = st.slider("Places de parking", 0, 3, 1)

with col2:
    st.subheader("Équipements")
    mainroad        = st.selectbox("Route principale ?", ["yes", "no"])
    guestroom       = st.selectbox("Chambre d'amis ?", ["yes", "no"])
    basement        = st.selectbox("Sous-sol ?", ["yes", "no"])
    hotwaterheating = st.selectbox("Chauffage eau chaude ?", ["yes", "no"])

with col3:
    st.subheader("Autres critères")
    airconditioning  = st.selectbox("Climatisation ?", ["yes", "no"])
    prefarea         = st.selectbox("Zone privilégiée ?", ["yes", "no"])
    furnishing_label = st.selectbox("Ameublement", ["Meublée", "Semi-meublée", "Non meublée"])
    furnishing_map   = {"Meublée": 2, "Semi-meublée": 1, "Non meublée": 0}
    furnishingstatus = furnishing_map[furnishing_label]

st.divider()

def yn(v): return 1 if v == "yes" else 0

if st.button("🔮 Estimer le prix", use_container_width=True, type="primary"):
    input_data = pd.DataFrame([{
        "AREA": area, "BEDROOMS": bedrooms, "BATHROOMS": bathrooms,
        "STORIES": stories, "MAINROAD": yn(mainroad),
        "GUESTROOM": yn(guestroom), "BASEMENT": yn(basement),
        "HOTWATERHEATING": yn(hotwaterheating),
        "AIRCONDITIONING": yn(airconditioning), "PARKING": parking,
        "PREFAREA": yn(prefarea), "FURNISHINGSTATUS": furnishingstatus
    }])

    input_sp       = session.create_dataframe(input_data)
    result_sp      = model.run(input_sp, function_name="predict")
    predicted_price = result_sp.to_pandas().iloc[0, -1]

    st.success(f"💰 Prix estimé : **{predicted_price:,.0f} €**")
    st.subheader("Récapitulatif")
    st.dataframe(input_data, use_container_width=True)